### 1. Calculo de metricas
Para este notebook se requiere utilizar hasta Python 3.10, para que la libreria AligScore se deben contar con versiones especificas que pueden hacer funcionar mal los procesos de fine tuning por lo que se dejan por separado, la idea es tomar todos los archivos csv que cuentan con el texto cientifico, texto resumen original y el resumen generado por medio del LLM en este notebook y realizar el calculo de las metricas: Legibilidad, Relevancia y Factualidad.

Instalacion libreria AlignScore:
https://github.com/yuh-zha/AlignScore

In [1]:
!pip install --quiet  -r req-fine-models-metrics.txt

In [2]:
import pandas as pd
import numpy as np
import textstat
from typing import List, Dict, Any, Optional, Tuple
from bert_score import score as bert_score
import torch
from pathlib import Path
DATA_ALIGN = Path("./models/alignscore")
DATA_ALIGN.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
### Modelo requerido base, puede utilizarse large tambien, podria descargarse de HuggingFace, en una proxima revision lo ajusto.

!(cd models/alignscore; curl -O -OL https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt)
#!(cd models/alignscore; curl -O -OL https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-large.ckpt)


In [3]:
def calcular_factualidad_alignscore(preds, refs,evaluation_mode, batch_size, device,flag_threshold: float = 0.5):
#evaluation_mode,    # 'nli_sp' (por defecto AlignScore), 'nli', 'bin_sp', 'bin'
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"

    # Import tardío para que esta función siga importando aunque no esté instalada la lib.
    from alignscore import AlignScore  
    # Inicializar scorer
    backbone = 'roberta-base'
    scorer = AlignScore(
        model="roberta-base",
        batch_size=batch_size,
        device=device,
        ckpt_path='models/alignscore/AlignScore-base.ckpt',
        evaluation_mode=evaluation_mode
    )

    scores = scorer.score(contexts=refs, claims=preds) 
    scores = [float(s) for s in scores]

    flags_low = [bool(s < flag_threshold) for s in scores]
    per_example = pd.DataFrame({"alignscore": scores,"flag_low": flags_low}) 

    summary = {
        "mean_alignscore": float(np.mean(scores)) if scores else float("nan"),
        "std_alignscore":  float(np.std(scores)) if scores else float("nan"),
        "min_alignscore":  float(np.min(scores)) if scores else float("nan"),
        "max_alignscore":  float(np.max(scores)) if scores else float("nan"),
        "n_examples":      int(len(scores)),
        "backbone":        backbone,
        "evaluation_mode": evaluation_mode,
        "batch_size":      int(batch_size),
        "device":          device,
        "ckpt_path":       'models/alignscore/AlignScore-base.ckpt',
        "flag_threshold":  float(flag_threshold)
    }

    return summary, per_example



In [4]:
def calcular_bertscore_relevancia(preds,refs,idf,rescale_with_baseline,batch_size,device):

    modelo = "roberta-base"
    assert len(preds) == len(refs), "preds y refs deben tener la misma longitud"


    P, R, F1 = bert_score(
        cands=preds.tolist(),
        refs=refs.tolist(),
        lang='en',
        model_type=modelo,
        idf=idf,
        rescale_with_baseline=rescale_with_baseline,
        batch_size=batch_size,
        device=device
    )

    p_list = [float(p) for p in P]
    r_list = [float(r) for r in R]
    f1_list = [float(f) for f in F1]

    summary = {
        "mean_precision": float(np.mean(p_list)) if p_list else float("nan"),
        "mean_recall":    float(np.mean(r_list)) if r_list else float("nan"),
        "mean_f1":        float(np.mean(f1_list)) if f1_list else float("nan"),
        "backbone_for_bertscore": modelo,
        "idf": bool(idf),
        "rescale_with_baseline": bool(rescale_with_baseline),
        "batch_size": int(batch_size),
        "device": device if device is not None else "auto"
    }

    per_example = {
        "bertscore_precision": p_list,
        "bertscore_recall": r_list,
        "bertscore_f1": f1_list
    }

    per_example = pd.DataFrame(per_example)

    return summary, per_example


In [5]:
def calcular_legibilidad_textstat(preds):
    lang = 'en'
    textstat.set_lang(lang)
    rows = []
    for t in preds:
        t = t or ""

        row = {
            "flesch_reading_ease":  float(textstat.flesch_reading_ease(t)),
            "flesch_kincaid_grade": float(textstat.flesch_kincaid_grade(t)),
        }
        row.update({
            "gunning_fog":              float(textstat.gunning_fog(t)),
            "smog_index":               float(textstat.smog_index(t)) if textstat.sentence_count(t) >= 3 else float("nan"),
            "dale_chall":               float(textstat.dale_chall_readability_score(t)),
            "automated_readability":    float(textstat.automated_readability_index(t)),
            "coleman_liau":             float(textstat.coleman_liau_index(t)),
            "text_standard":            textstat.text_standard(t, float_output=True),
            "num_sentences":            int(textstat.sentence_count(t)),
            "num_words":                int(textstat.lexicon_count(t, removepunct=True)),
            "syllables":                int(textstat.syllable_count(t)),
            "reading_time_sec":         float(textstat.reading_time(t)),
        })

        rows.append(row)

    def _try_mean(key: str):
        vals = [r[key] for r in rows if key in r and isinstance(r[key], (int, float)) and not np.isnan(r[key])]
        return float(np.mean(vals)) if vals else float("nan")

    keys = sorted({k for r in rows for k in r.keys()})
    summary = {"n_examples": len(preds), "lang": lang}
    for k in keys:
        summary[f"mean_{k}"] = _try_mean(k)

    per_example = pd.DataFrame(rows) 
    return summary, per_example


In [6]:
def calcular_metricas(df):
    print('**** Inicio relevancia')
    summary_relevancia, per_example_relevancia = calcular_bertscore_relevancia(
        df['gen_summary'],df['article'],
        idf=True,#Set pequeno a false, sino dejar en TRUE
        rescale_with_baseline=False,
        batch_size=1,
        device=device
    )
    print('**** Fin relevancia')
    print('**** Inicio legibilidad')
    summary_legibilidad, per_example_legibilidad = calcular_legibilidad_textstat(df['gen_summary'])
    print('**** Fin legibilidad')
    print('**** Inicio factualidad')
    summary_factualidad, per_example_factualidad = calcular_factualidad_alignscore(df['gen_summary'], df['article'],'nli_sp', 16, device)
    print('**** Fin factualidad')
    return summary_relevancia, summary_legibilidad,summary_factualidad


### Calculo de ejemplo de las 3 metricas requeridas para llama3

In [38]:
data = pd.read_csv('models/results/summaries_llama3.2-1b.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, 

**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file models/alignscore/AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/htorre/Documents/anaconda/anaconda3/envs/P310/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:255: UserWarning: Found keys that are not in the model state dict b

**** Fin factualidad


In [39]:
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)

{'mean_precision': 0.8279218648543144, 'mean_recall': 0.8130444171875322, 'mean_f1': 0.8203009269482857, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cpu')}
{'n_examples': 379, 'lang': 'en', 'mean_automated_readability': 12.962654002661017, 'mean_coleman_liau': 13.544975841342726, 'mean_dale_chall': 11.042229873784924, 'mean_flesch_kincaid_grade': 11.490371639590998, 'mean_flesch_reading_ease': 43.18348842750842, 'mean_gunning_fog': 13.882925550125412, 'mean_num_sentences': 20.094986807387862, 'mean_num_words': 334.712401055409, 'mean_reading_time_sec': 26.894172928759897, 'mean_smog_index': 13.303621532167545, 'mean_syllables': 576.4353562005277, 'mean_text_standard': 12.546174142480211}
{'mean_alignscore': 0.3286242190954868, 'std_alignscore': 0.08548606122105906, 'min_alignscore': 0.11750971525907516, 'max_alignscore': 0.5550947189331055, 'n_examples': 379, 'backbone': 'roberta-base', 'evaluation_mode'

In [40]:
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_llama3.2-1b.csv", index=False)

### Calculo de ejemplo de las 3 metricas requeridas para gemma

In [7]:
data = pd.read_csv('models/results/summaries_gemma-3-1b-pt.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/metrics_gemma-3-1b-pt.csv", index=False)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


/Users/htorre/Documents/anaconda/anaconda3/envs/P310/lib/python3.10/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file models/alignscore/AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized from the

**** Fin factualidad
{'mean_precision': 0.82111682938902, 'mean_recall': 0.8138819740006799, 'mean_f1': 0.8173210584803632, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cpu')}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 13.615634061159804, 'mean_coleman_liau': 14.088285805376556, 'mean_dale_chall': 11.466279532340204, 'mean_flesch_kincaid_grade': 11.819987138127718, 'mean_flesch_reading_ease': 40.55993701470229, 'mean_gunning_fog': 14.316384082759539, 'mean_num_sentences': 19.839473684210525, 'mean_num_words': 320.62105263157895, 'mean_reading_time_sec': 26.70564684210526, 'mean_smog_index': 13.536725684511072, 'mean_syllables': 565.2342105263158, 'mean_text_standard': 12.952631578947368}
{'mean_alignscore': 0.2831807450636437, 'std_alignscore': 0.0843669348098076, 'min_alignscore': 0.11659961938858032, 'max_alignscore': 0.6092774271965027, 'n_examples': 380, 'backbone': 'roberta-base'

### Calculo de ejemplo de las 3 metricas requeridas para qwen

In [8]:
data = pd.read_csv('models/results/summaries_qwen3.csv')
sum_relevancia, sum_legibilidad,sum_factualidad = calcular_metricas(data)
print(sum_relevancia)
print(sum_legibilidad)
print(sum_factualidad)
metricas = pd.concat({
    "relevancia": pd.json_normalize(sum_relevancia, sep="__"),
    "legibilidad": pd.json_normalize(sum_legibilidad, sep="__"),
    "factualidad": pd.json_normalize(sum_factualidad, sep="__"),
}, axis=1)

metricas.to_csv("models/results/summaries_qwen3.csv", index=False)

**** Inicio relevancia


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid dead

**** Fin relevancia
**** Inicio legibilidad
**** Fin legibilidad
**** Inicio factualidad


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file models/alignscore/AlignScore-base.ckpt`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Users/htorre/Documents/anaconda/anaconda3/envs/P310/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:255: UserWarning: Found keys that are not in the model state dict b

**** Fin factualidad
{'mean_precision': 0.8174097315261238, 'mean_recall': 0.8262641431469666, 'mean_f1': 0.8217499783164577, 'backbone_for_bertscore': 'roberta-base', 'idf': True, 'rescale_with_baseline': False, 'batch_size': 1, 'device': device(type='cpu')}
{'n_examples': 380, 'lang': 'en', 'mean_automated_readability': 15.004022756981549, 'mean_coleman_liau': 15.998775581095632, 'mean_dale_chall': 12.185788195727548, 'mean_flesch_kincaid_grade': 13.416241243867473, 'mean_flesch_reading_ease': 29.240736028026042, 'mean_gunning_fog': 16.60301638393216, 'mean_num_sentences': 49.09736842105263, 'mean_num_words': 793.2552631578948, 'mean_reading_time_sec': 69.35415739473684, 'mean_smog_index': 15.019013091117023, 'mean_syllables': 1498.8315789473684, 'mean_text_standard': 14.33157894736842}
{'mean_alignscore': 0.34249391161689635, 'std_alignscore': 0.11920498883152891, 'min_alignscore': 0.11683043092489243, 'max_alignscore': 0.8931159973144531, 'n_examples': 380, 'backbone': 'roberta-bas